# Sleep Quality to Output Model Analysis

This notebook tests whether **sleep quality** can support the output side of the fixed installation path.

It uses dyslexia-friendly reporting:

- short sections
- simple tables
- clear yes/no decisions

The notebook does **not** use `sleep_score` as evidence.
It only claims support when a model beats the baseline mean model on test error.

In [1]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=UserWarning)

CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR if (CURRENT_DIR / 'model_results_summary.md').exists() else CURRENT_DIR.parent
DATASET_NAME = 'sleep_deprivation_dataset_detailed.csv'
GRAPHS_DIR = PROJECT_DIR / 'Graphs'
GRAPHS_DIR.mkdir(exist_ok=True)

print('Project folder:', PROJECT_DIR)
print('Graphs folder:', GRAPHS_DIR)

Project folder: /workspace/Sleep-datasets-official
Graphs folder: /workspace/Sleep-datasets-official/Graphs


## 1. Find the dataset

We search recursively from the project folder for:

```text
sleep_deprivation_dataset_detailed.csv
```

In [2]:
matches = sorted(PROJECT_DIR.rglob(DATASET_NAME))
print(f'Matches found: {len(matches)}')
for path in matches:
    print(path.relative_to(PROJECT_DIR))

if not matches:
    print('Dataset was not found. The requested output-side models cannot be tested yet.')

Matches found: 0
Dataset was not found. The requested output-side models cannot be tested yet.


In [3]:
if matches:
    dataset_path = matches[0]
    df = pd.read_csv(dataset_path)
    print('Using:', dataset_path.relative_to(PROJECT_DIR))
    print('Shape:', df.shape)
    display(df.head())
    display(pd.DataFrame({'column': df.columns}))
else:
    dataset_path = None
    df = pd.DataFrame()

## 2. Model paths

Predictors for every path:

```text
Sleep_Quality_Score + Sleep_Hours + Daytime_Sleepiness
```

Outputs tested:

| path | output column | installation output |
|---|---|---|
| A | `N_Back_Accuracy` | memory |
| B | `PVT_Reaction_Time` | reaction time |
| C | `Stroop_Task_Reaction_Time` | processing speed / attention |
| D | `Stress_Level` | stress |
| E | `Emotion_Regulation_Score` | mood / emotion regulation |

In [4]:
PREDICTORS = ['Sleep_Quality_Score', 'Sleep_Hours', 'Daytime_Sleepiness']
TARGETS = {
    'A': ('N_Back_Accuracy', 'memory'),
    'B': ('PVT_Reaction_Time', 'reaction time'),
    'C': ('Stroop_Task_Reaction_Time', 'processing speed / attention'),
    'D': ('Stress_Level', 'stress'),
    'E': ('Emotion_Regulation_Score', 'mood / emotion regulation'),
}

def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))

def evaluate_target(data, target_col, output_name):
    needed = PREDICTORS + [target_col]
    missing = [c for c in needed if c not in data.columns]
    if missing:
        return {'output': output_name, 'target': target_col, 'status': 'missing columns', 'missing_columns': ', '.join(missing)}

    model_df = data[needed].apply(pd.to_numeric, errors='coerce').dropna()
    rows_used = len(model_df)
    if rows_used < 10:
        return {'output': output_name, 'target': target_col, 'status': 'not enough rows', 'rows_used': rows_used}

    X = model_df[PREDICTORS]
    y = model_df[target_col]
    test_size = 0.25 if rows_used >= 20 else 0.3
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)

    baseline = DummyRegressor(strategy='mean')
    linear = make_pipeline(StandardScaler(), LinearRegression())
    forest = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42)
    models = {'baseline_mean': baseline, 'linear_regression': linear, 'random_forest_comparison': forest}

    result = {'output': output_name, 'target': target_col, 'status': 'tested', 'rows_used': rows_used}
    predictions = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)
        predictions[name] = test_pred
        result[f'{name}_train_r2'] = r2_score(y_train, train_pred)
        result[f'{name}_test_r2'] = r2_score(y_test, test_pred)
        result[f'{name}_test_mae'] = mean_absolute_error(y_test, test_pred)
        result[f'{name}_test_rmse'] = rmse(y_test, test_pred)

    lin_model = linear.named_steps['linearregression']
    result['linear_intercept'] = lin_model.intercept_
    result['linear_formula'] = ' + '.join([f'{coef:.4f} * z({name})' for coef, name in zip(lin_model.coef_, PREDICTORS)])
    result['linear_coefficients'] = dict(zip(PREDICTORS, lin_model.coef_))
    result['forest_importance'] = dict(zip(PREDICTORS, forest.feature_importances_))

    base_rmse = result['baseline_mean_test_rmse']
    lin_rmse = result['linear_regression_test_rmse']
    rf_rmse = result['random_forest_comparison_test_rmse']
    result['best_model'] = 'linear_regression' if lin_rmse <= rf_rmse else 'random_forest_comparison'
    result['best_test_rmse'] = min(lin_rmse, rf_rmse)
    result['beats_baseline'] = result['best_test_rmse'] < base_rmse
    result['decision'] = 'use' if result['beats_baseline'] else 'do not use'

    fig, ax = plt.subplots(figsize=(4.5, 3.2))
    metrics = [base_rmse, lin_rmse, rf_rmse]
    ax.bar(['baseline', 'linear', 'random forest'], metrics, color=['#999999', '#4C78A8', '#F58518'])
    ax.set_title(f'{output_name}: test RMSE')
    ax.set_ylabel('RMSE')
    ax.tick_params(axis='x', rotation=20)
    fig.tight_layout()
    graph_path = GRAPHS_DIR / f'sleep_quality_to_{target_col}_rmse.png'
    fig.savefig(graph_path, dpi=150)
    plt.close(fig)
    result['graph'] = str(graph_path.relative_to(PROJECT_DIR))
    return result

results = []
if dataset_path is not None:
    for path_id, (target, output) in TARGETS.items():
        row = evaluate_target(df, target, output)
        row['path'] = path_id
        results.append(row)
else:
    for path_id, (target, output) in TARGETS.items():
        results.append({'path': path_id, 'output': output, 'target': target, 'status': 'dataset not found', 'decision': 'do not use'})

results_df = pd.DataFrame(results)
display(results_df)

,path,output,target,status,decision
0,A,memory,N_Back_Accuracy,dataset not found,do not use
1,B,reaction time,PVT_Reaction_Time,dataset not found,do not use
2,C,processing speed / attention,Stroop_Task_Reaction_Time,dataset not found,do not use
3,D,stress,Stress_Level,dataset not found,do not use
4,E,mood / emotion regulation,Emotion_Regulation_Score,dataset not found,do not use


## 3. Decision rule

A path is marked **use** only when the best tested model has lower test RMSE than the baseline mean model.

If the dataset or required columns are missing, the path is **do not use**.

In [5]:
if not results_df.empty:
    display_cols = [c for c in ['path','output','target','status','rows_used','baseline_mean_test_r2','baseline_mean_test_mae','baseline_mean_test_rmse','linear_regression_train_r2','linear_regression_test_r2','linear_regression_test_mae','linear_regression_test_rmse','random_forest_comparison_train_r2','random_forest_comparison_test_r2','random_forest_comparison_test_mae','random_forest_comparison_test_rmse','best_model','beats_baseline','decision','graph'] if c in results_df.columns]
    display(results_df[display_cols])

    out_path = PROJECT_DIR / 'sleep_quality_output_model_results.csv'
    results_df.to_csv(out_path, index=False)
    print('Saved results table:', out_path.relative_to(PROJECT_DIR))

,path,output,target,status,decision
0,A,memory,N_Back_Accuracy,dataset not found,do not use
1,B,reaction time,PVT_Reaction_Time,dataset not found,do not use
2,C,processing speed / attention,Stroop_Task_Reaction_Time,dataset not found,do not use
3,D,stress,Stress_Level,dataset not found,do not use
4,E,mood / emotion regulation,Emotion_Regulation_Score,dataset not found,do not use


Saved results table: sleep_quality_output_model_results.csv
